# Bash Fundamentals

## Quick notes

### What this covers
The ground floor of every bash script: how a script gets run (shebang, `bash script`, `./script`, `source`), variables, the three kinds of quoting (plus `$'...'`), word splitting and globbing, command substitution, arithmetic, exit statuses, `readonly`/`export`/`unset`, special parameters, finding out what a name really is (`type`), the order the shell expands things, line continuation, command lists and `set -x` tracing.

### Key points
- The shebang (`#!/usr/bin/env bash`) only matters when you run the file directly (`./script`). `bash script` ignores it. `source script` runs the file inside your current shell.
- Assignment has no spaces: `x=5`. `x = 5` runs a command called `x`.
- Variables are strings. Numbers are just strings that arithmetic can read.
- Quote every expansion (`"$x"`, `"$(cmd)"`, `"$@"`) unless you really want splitting and globbing.
- Single quotes are fully literal. Double quotes allow `$`, backticks and `\`. `$'...'` understands escapes like `\n` and `\t`.
- Every command has an exit status. `0` is success, anything else is failure. `$?` holds the last one.
- `export` puts a variable in the environment so child processes see it. A plain shell variable stays in this shell only.
- The shell expands a command line in a fixed order: braces, tilde, parameters / arithmetic / command substitution, word splitting, filename globbing, quote removal.

### Gotchas (the part worth revising twice)
- An unquoted `$var` is split on whitespace and then globbed. A file called `my file.txt` becomes two words.
- A glob that matches nothing is left as the literal pattern (unless you set `nullglob`).
- `$?` is overwritten by the next command, even by `echo`. Save it right away if you need it.
- `a && b || c` is NOT if/else: if `b` fails, `c` runs too.
- A script run with `bash script.sh` cannot change your current directory or variables. `source` can.
- `$(( ))` is integer-only: `7 / 2` is `3`.
- `which` is an external program and does not know about builtins, functions or aliases. `type` and `command -v` do.

### When it matters
Everything. Quoting discipline is what separates a script that works on your test files from one that survives a filename with a space in it.

### One-line recap example
```bash
name="Bash Script"
echo "$name"   # safe: one word, spacing kept
echo $name     # risky: split into two words
```

Related notebooks: `02-Control-Flow` (tests and loops build on exit statuses), `03-Functions-And-Arguments`, `04-Strings-And-Arrays`.

## 01. The shebang line
- The first line `#!/usr/bin/env bash` tells the kernel which interpreter should run the file when you execute it directly.
- `env bash` looks `bash` up on `PATH`. This is more portable than hard-coding `/bin/bash` (Homebrew bash on macOS lives elsewhere, for example).
- The shebang is just a comment to bash itself. It only acts when the OS starts the file for you.
- On this machine `/bin/bash` is bash 3.2, which is very old. Features from bash 4+ are shown as "not run here" in these notebooks.

In [1]:
# setup: a %%bash cell magic that keeps ONE bash session alive between cells
import atexit, os, shutil, subprocess, tempfile, uuid
from IPython.core.magic import register_cell_magic

_WORK = tempfile.mkdtemp(prefix="bashnb_")
atexit.register(shutil.rmtree, _WORK, ignore_errors=True)
_ENV = {k: v for k, v in os.environ.items() if k not in ("CLICOLOR", "CLICOLOR_FORCE", "FORCE_COLOR", "PYDEVD_DISABLE_FILE_VALIDATION")}
_ENV.update(LC_ALL="C", TZ="UTC", HOME=_WORK, PS1="", PS4="+ ")
_session = None

def _start():
    global _session
    _session = subprocess.Popen(["bash", "--noprofile", "--norc"], stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1, cwd=_WORK, env=_ENV)

def _clean(text):
    return text.replace(os.path.join(_WORK, "script"), "script").replace(_WORK, "<work>")

def _run_persistent(code, timeout=60):
    global _session
    if _session is None or _session.poll() is not None:
        _start()
    token = uuid.uuid4().hex
    path = os.path.join(_WORK, "script")
    with open(path, "w") as fh:
        fh.write(code + "\n")
    _session.stdin.write(". '%s' </dev/null; printf '\\n%s %%d\\n' $?\n" % (path, token))
    _session.stdin.flush()
    out, status = [], None
    import threading
    result = {}
    def reader():
        while True:
            line = _session.stdout.readline()
            if not line:
                result["status"] = None; return
            if line.startswith(token + " "):
                result["status"] = int(line.split()[1]); return
            out.append(line)
    t = threading.Thread(target=reader, daemon=True); t.start(); t.join(timeout)
    os.remove(path)
    if t.is_alive():
        _session.kill(); _session = None
        return "".join(out) + "[timed out]\n", 124
    if result.get("status") is None:       # the shell itself exited (e.g. the cell ran `exit N`)
        code = _session.wait(); _session = None
        return "".join(out) + "(the bash session ended; the next cell starts a fresh one)\n", code
    text = "".join(out)
    if text.endswith("\n\n"):
        text = text[:-1]          # drop the blank line our status marker adds
    return text, result.get("status") if result.get("status") is not None else 1

def _run_fresh(code, timeout=60):
    path = os.path.join(_WORK, "script")
    with open(path, "w") as fh:
        fh.write(code + "\n")
    done = subprocess.run(["bash", "--noprofile", "--norc", path], stdin=subprocess.DEVNULL, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, cwd=_WORK, env=_ENV, timeout=timeout)
    os.remove(path)
    return done.stdout, done.returncode

@register_cell_magic
def bash(line, cell):
    """Run the cell in bash. Flags: fresh (new shell), expect_fail (show non-zero status instead of raising)."""
    flags = line.split()
    text, status = (_run_fresh if "fresh" in flags else _run_persistent)(cell)
    print(_clean(text), end="" if text.endswith("\n") or not text else "\n")
    if status != 0:
        if "expect_fail" in flags:
            print("[exit status %d]" % status)
        else:
            raise RuntimeError("bash cell exited with status %d" % status)

In [2]:
%%bash
echo "major version: ${BASH_VERSINFO[0]}"
echo "version line starts with: ${BASH_VERSION%%(*}"

major version: 3
version line starts with: 3.2.57


## 02. `bash script` vs `./script` vs `source script`
- `bash script` starts a new bash process and ignores the shebang. The file does not need the execute bit.
- `./script` asks the OS to run the file, so it needs `chmod +x`, and the shebang picks the interpreter.
- `source script` (or `. script`) runs the lines inside your current shell. Variables, `cd` and functions it sets stay afterwards.
- The first two run in a child process, so nothing the script does to variables or the directory comes back.

In [3]:
%%bash
printf '%s\n' '#!/usr/bin/env bash' 'greeting="set by the script"' 'cd /' 'echo "inside: pwd=$(pwd)"' > demo.sh

In [4]:
%%bash expect_fail
echo "no exec bit yet:"
./demo.sh

no exec bit yet:
script: line 2: ./demo.sh: Permission denied
[exit status 126]


In [5]:
%%bash
bash demo.sh                       # works without the exec bit
echo "after bash demo.sh: greeting=[${greeting:-}]  back in work dir: $(basename "$(pwd)" | sed 's/^bashnb_.*/<work>/')"
chmod +x demo.sh
./demo.sh                          # now it runs via its shebang
echo "after ./demo.sh: greeting=[${greeting:-}]"
. ./demo.sh                        # same as: source ./demo.sh
echo "after source: greeting=[${greeting:-}], pwd=$(pwd)"
cd "$HOME"                         # go back to the work directory

inside: pwd=/
after bash demo.sh: greeting=[]  back in work dir: <work>
inside: pwd=/
after ./demo.sh: greeting=[]
inside: pwd=/
after source: greeting=[set by the script], pwd=/


## 03. Variables: assignment and expansion
- No spaces around `=`. `x = 5` is the command `x` with two arguments, `=` and `5`.
- All variables are strings by default.
- `$x` and `${x}` are the same. Use braces when the name would run into following text: `${name}Script`.
- Reading an unset variable gives an empty string (unless `set -u` is on, see `04-Scripting-Craft/01-Advanced-Bash`).

In [6]:
%%bash
x=5
name="Bash"
echo "x is $x, name is ${name}"
echo "with braces: ${name}Script"
echo "without braces looks for a variable named nameScript: [$nameScript]"

x is 5, name is Bash
with braces: BashScript
without braces looks for a variable named nameScript: []


In [7]:
%%bash expect_fail
y = 5

script: line 1: y: command not found
[exit status 127]


In [8]:
%%bash
# a one-shot assignment before a command applies only to that command
greeting=hello
greeting=hi bash -c 'echo "child sees: $greeting"'
echo "shell still has: $greeting"

child sees: hi
shell still has: hello


## 04. Braces and parameter expansion basics
- `${var:-default}` gives a default when `var` is unset or empty (it does not assign).
- `${var:=default}` assigns the default too.
- `${#var}` is the length, `${var%suffix}` strips a suffix, `${var#prefix}` strips a prefix.
- These all work in bash 3.2 and in plain POSIX sh. Case conversion (`${var^^}`) needs bash 4+. See `04-Strings-And-Arrays` for more.

In [9]:
%%bash
unset maybe
echo "default: ${maybe:-fallback}   still unset? [${maybe-unset}]"
echo "assign:  ${maybe:=fallback}   now set to [$maybe]"
file="archive.tar.gz"
echo "length: ${#file}   no .gz: ${file%.gz}   no prefix: ${file#archive.}"

default: fallback   still unset? [unset]
assign:  fallback   now set to [fallback]
length: 14   no .gz: archive.tar   no prefix: tar.gz


```bash
# needs bash 4+ (not run here)
word=hello
echo "${word^^}"   # HELLO
```

## 05. Quoting in depth
- No quotes: the shell splits the result of expansions on whitespace and expands globs.
- Double quotes `"..."`: `$var`, `$(cmd)`, `$((expr))` and backticks still work, but the result stays one word. `\` only escapes `$`, backtick, `"`, `\` and newline.
- Single quotes `'...'`: everything is literal. You cannot put a `'` inside them.
- `$'...'` (ANSI-C quoting) is single-quote-like but turns `\n`, `\t`, `\\`, `\'` into real characters.
- A backslash outside quotes escapes the next character.

In [10]:
%%bash
name="Bash"
echo 'single: $name $(echo hi) \n stay literal'
echo "double: $name $(echo hi) \$name stays literal when escaped"
echo $'ansi-c: a\tb and a quote \' inside'
printf '%s\n' $'line1\nline2'
echo backslash\ outside\ quotes

single: $name $(echo hi) \n stay literal
double: Bash hi $name stays literal when escaped
ansi-c: a	b and a quote ' inside
line1
line2
backslash outside quotes


In [11]:
%%bash
# gluing quote styles into one word
echo 'It'"'"'s a single quote inside single quotes'
echo "The cost is \$5 and the shell is $name"

It's a single quote inside single quotes
The cost is $5 and the shell is Bash


In [12]:
%%bash
# echo and escapes: the shell does NOT turn \n into a newline in plain quotes
echo "a\nb"
printf 'a\nb\n'           # printf interprets \n itself; prefer printf in scripts

a\nb
a
b


## 06. Word splitting: files with spaces
- After an unquoted expansion, the shell splits the result on characters in `IFS` (space, tab, newline by default).
- A filename with a space becomes two arguments. This is the classic bug.
- Quote it: `"$file"` stays one argument.

In [13]:
%%bash
touch "my file.txt" plain.txt
file="my file.txt"
echo "--- BAD: unquoted"
ls -d $file 2>&1 | sort
echo "--- GOOD: quoted"
ls "$file"
rm -f "my file.txt" plain.txt

--- BAD: unquoted
ls: file.txt: No such file or directory
ls: my: No such file or directory
--- GOOD: quoted
my file.txt


In [14]:
%%bash
# counting arguments makes the splitting visible
count() { echo "$# argument(s)"; }
v="a  b   c"
count $v
count "$v"

3 argument(s)
1 argument(s)


In [15]:
%%bash
# changing IFS changes where splitting happens (only matters for UNQUOTED expansions)
line="one:two:three"
old=$IFS
IFS=:
count $line
IFS=$old
count $line

3 argument(s)
1 argument(s)


## 07. Globbing gotchas
- `*`, `?` and `[abc]` are expanded by the shell into matching filenames, after variable expansion.
- A glob that matches nothing stays as the literal text. `ls *.zip` then fails with "No such file".
- Quotes switch globbing off: `"*.txt"` is just the characters `*.txt`.
- `*` does not match names starting with `.` (hidden files), and does not cross `/`.
- A variable holding `*.txt` expands as a glob when unquoted.

In [16]:
%%bash
touch a.txt b.txt .hidden.txt plain.txt
echo *.txt
echo "*.txt"
echo '*.txt'
echo *.zip             # no match: the pattern is left as is
echo .*txt             # hidden files need an explicit leading dot
pattern='*.txt'
echo "quoted:   $pattern"
echo "unquoted: " $pattern

a.txt b.txt plain.txt
*.txt
*.txt
*.zip
.hidden.txt
quoted:   *.txt
unquoted:  a.txt b.txt plain.txt


In [17]:
%%bash
# no match changes behavior of the command that receives the literal text
ls *.zip
echo "status: $?"

ls: *.zip: No such file or directory
status: 1


Note: `ls` exits with 1 here on macOS (BSD) but 2 on GNU/Linux. Do not depend on the exact number.

In [18]:
%%bash
# nullglob makes a no-match glob expand to nothing (works in bash 3.2)
shopt -s nullglob
echo "nullglob: [$(echo *.zip)]"
shopt -u nullglob
rm -f a.txt b.txt .hidden.txt plain.txt "my file.txt"

nullglob: []


## 08. Command substitution: `$(...)` vs backticks
- `$(cmd)` runs `cmd` and replaces itself with its standard output. Trailing newlines are removed.
- Backticks do the same, but nesting needs ugly escaping and the quoting rules are odd. Prefer `$(...)`.
- Always quote it (`"$(cmd)"`) unless you want splitting.
- The exit status of an assignment like `x=$(cmd)` is the status of `cmd`. In `local x=$(cmd)` or `export x=$(cmd)` the status is that of `local`/`export`, which hides failures.

In [19]:
%%bash
d=$(echo "fixed text")
old=`echo "fixed text"`
echo "$d / $old"
echo "nested easily: $(echo "outer $(echo inner)")"
echo "nested with backticks needs escapes: `echo "outer \`echo inner\`"`"

fixed text / fixed text
nested easily: outer inner
nested with backticks needs escapes: outer inner


In [20]:
%%bash
# trailing newlines are stripped, inner ones are kept
s=$(printf 'a\nb\n\n\n')
printf '[%s]\n' "$s"
# the status of an assignment is the status of the command inside
x=$(false); echo "x=\$(false) status: $?"

[a
b]
x=$(false) status: 1


In [21]:
%%bash
# a function that hides a failure: the status is local's, not false's
f() { local v=$(false); }
f
echo "local v=\$(false) -> status of f: $?   (failure hidden)"
g() { local v; v=$(false); }
g
echo "declare first, assign second -> status of g: $?   (failure visible)"

local v=$(false) -> status of f: 0   (failure hidden)
declare first, assign second -> status of g: 1   (failure visible)


## 09. Arithmetic: `$(( ))` and `(( ))`
- `$(( expr ))` is replaced by the integer result. Inside, variables need no `$`.
- `(( expr ))` is a command: it succeeds (status 0) when the result is non-zero, and fails (status 1) when the result is 0. It is used in `if` and loops.
- Careful: `(( a++ ))` returns status 1 when `a` was 0 (the old value, 0, is the result). On newer bash (4+), under `set -e` that kills the script. Use `a=$((a + 1))` there.
- Integers only: `7 / 2` is `3`. For decimals use `awk` or `bc`.
- Operators: `+ - * / % **`, `++ --`, `+= -=`, comparisons, `&& ||`.
- A value with a leading `0` is read as octal: `08` is an error. Force base 10 with `10#08`.

In [22]:
%%bash
a=7; b=2
echo "sum=$((a + b)) div=$((a / b)) mod=$((a % b)) pow=$((a ** b))"
(( a++ ))
echo "after (( a++ )): a=$a"
(( a += 10 )); echo "a=$a"
awk 'BEGIN { printf "decimal division: %.2f\n", 7 / 2 }'

sum=9 div=3 mod=1 pow=49
after (( a++ )): a=8
a=18
decimal division: 3.50


In [23]:
%%bash expect_fail
(( 0 ))
echo "(( 0 )) status: $?"
(( 5 > 3 )); echo "(( 5 > 3 )) status: $?"
(( 1 - 1 ))

(( 0 )) status: 1
(( 5 > 3 )) status: 0
[exit status 1]


In [24]:
%%bash expect_fail
echo "base 10 forced: $(( 10#08 + 1 ))"
echo $(( 08 + 1 ))

base 10 forced: 9
script: line 2: 08: value too great for base (error token is "08")
[exit status 1]


## 10. Exit status: `$?` and `exit`
- Every command ends with a status from 0 to 255. `0` means success.
- `$?` is the status of the most recent command. It changes after every command, including `echo`.
- `exit N` ends the shell (or script) with status N. A bare `exit` uses the last status.
- Conventions: 1 general error, 2 misuse of a builtin / usage error, 126 found but not executable, 127 command not found, 128+N killed by signal N.
- `!` in front of a command negates its status.

In [25]:
%%bash
true;  echo "true -> $?"
false; echo "false -> $?"
grep -q xyz <<< "abc"; echo "grep no match -> $?"
nosuchcommand_xyz 2>/dev/null; echo "not found -> $?"
sh -c 'exit 7'; echo "exit 7 -> $?"
! false; echo "! false -> $?"

true -> 0
false -> 1
grep no match -> 1
not found -> 127
exit 7 -> 7
! false -> 0


In [26]:
%%bash
# $? is overwritten immediately
false
echo "first echo sees: $?"
echo "second echo sees: $?"
false
rc=$?                     # save it if you need it later
echo "saved: $rc"

first echo sees: 1
second echo sees: 0
saved: 1


In [27]:
%%bash fresh expect_fail
echo "this script ends with its own status"
exit 3

this script ends with its own status
[exit status 3]


## 11. `readonly`, `export`, `unset`, `declare`
- `readonly NAME=value` makes a constant. Changing or unsetting it is an error.
- `export NAME` puts the variable in the environment of child processes.
- `unset NAME` removes a variable (or function with `unset -f`).
- `declare -p NAME` shows how a variable is stored (value and attributes). `declare -i` makes assignments evaluate as arithmetic.

In [28]:
%%bash
readonly PI=3.14159
echo "PI=$PI"
( PI=3 ) 2>&1 | sed 's/^[^:]*: //'
( unset PI ) 2>&1 | sed 's/^[^:]*: //'
declare -p PI

PI=3.14159
line 3: PI: readonly variable
line 4: unset: PI: cannot unset: readonly variable
declare -r PI="3.14159"


In [29]:
%%bash
unset PI 2>/dev/null; echo "still there after failed unset: $PI"
declare -i n=2+3
echo "declare -i n=2+3 gives $n"
m=2+3
echo "plain m=2+3 gives $m"
unset n m
echo "after unset: [${n-gone}] [${m-gone}]"

still there after failed unset: 3.14159
declare -i n=2+3 gives 5
plain m=2+3 gives 2+3
after unset: [gone] [gone]


## 12. Environment variables vs shell variables
- A shell variable lives only in the current shell. An exported variable is copied into the environment of every child process.
- The copy goes one way: children cannot change the parent's variables.
- `VAR=value command` sets it only for that one command.
- `env` or `printenv` list the environment. Use `export -p` for what this shell would pass on.

In [30]:
%%bash
shellvar="shell only"
export envvar="exported"
bash -c 'echo "child sees shellvar=[${shellvar-unset}] envvar=[${envvar-unset}]"'
bash -c 'envvar=changed-in-child; echo "child changed it"'
echo "parent still has: $envvar"
printenv envvar
printenv shellvar || echo "printenv shellvar: not in the environment (status $?)"

child sees shellvar=[unset] envvar=[exported]
child changed it
parent still has: exported
exported
printenv shellvar: not in the environment (status 1)


In [31]:
%%bash
# export after assignment works too, and unexport with export -n
later="x"
export later
export -n later
bash -c 'echo "after export -n: [${later-unset}]"'
unset shellvar envvar later

after export -n: [unset]


## 13. Special parameters
- `$0` name of the script, `$1`..`$9`, `${10}` positional arguments, `$#` how many.
- `"$@"` every argument as a separate word (quote it). `"$*"` all arguments joined into one word.
- `$?` last status, `$$` PID of the shell, `$!` PID of the last background job, `$_` last argument of the previous command, `$-` current option flags.
- PIDs differ on every run, so below we only print things that are always the same, such as whether `$!` is set.

In [32]:
%%bash
cat > args.sh <<'SH'
echo "name: $0"
echo "count: $#"
echo "first: $1  second: $2"
for a in "$@"; do echo "  \"\$@\" item: [$a]"; done
for a in "$*"; do echo "  \"\$*\" item: [$a]"; done
SH
bash args.sh one "two words" three

name: args.sh
count: 3
first: one  second: two words
  "$@" item: [one]
  "$@" item: [two words]
  "$@" item: [three]
  "$*" item: [one two words three]


In [33]:
%%bash
echo alpha beta >/dev/null
echo "\$_ is the last argument of the previous command: $_"
case $$ in ''|*[!0-9]*) echo "\$\$ is not a number" ;; *) echo "\$\$ is the shell's PID (a number)" ;; esac
sleep 0 &
echo "\$! is set after starting a job: ${!:+yes}"
wait
case $- in *i*) echo "interactive" ;; *) echo "non-interactive shell (no i in \$-)" ;; esac
rm -f args.sh

$_ is the last argument of the previous command: beta
$$ is the shell's PID (a number)
$! is set after starting a job: yes
non-interactive shell (no i in $-)


`$0` inside a sourced file or a notebook cell is the name of the *shell*, not the file, so it is not printed here. When you run `bash args.sh`, `$0` is `args.sh`.

## 14. Comments
- `#` starts a comment, but only at the start of a word. `a#b` is just the text `a#b`.
- There is no block comment. A common trick is a here-document sent to `:` (a no-op).
- A `#` inside quotes is a normal character.

In [34]:
%%bash
echo "runs"      # trailing comment
echo a#b "#not a comment" '#neither'
: <<'COMMENT'
This whole block is ignored.
It is a here-document handed to the no-op command.
COMMENT
echo "after the block"

runs
a#b #not a comment #neither
after the block


## 15. `type`, `command -v` and `which`
- `type name` says what kind of thing `name` is: alias, keyword, function, builtin or file. `type -a` lists all matches. `type -t` prints just the kind.
- `command -v name` prints how the shell would run it, and has a failing status if not found. It is the portable (POSIX) way to test for a tool: `command -v jq >/dev/null`.
- `which` is a separate program. It only searches `PATH`, does not know functions or builtins, and behaves differently between systems (GNU `which`, BSD `which`, and some distros print nothing or an error).

In [35]:
%%bash
hello() { :; }
for n in cd if hello ls nosuchthing_xyz; do
  printf '%-16s type -t: %s\n' "$n" "$(type -t "$n" || echo '(none)')"
done

cd               type -t: builtin
if               type -t: keyword
hello            type -t: function
ls               type -t: file
nosuchthing_xyz  type -t: (none)


In [36]:
%%bash
type -a cd | sed 's|/.*|<path>|'      # builtin AND an external file exist
command -v hello                       # a function: prints its name
command -v nosuchthing_xyz || echo "command -v: not found, status $?"
case "$(command -v ls)" in /*) echo "command -v ls gives a path" ;; esac
case "$(which hello)" in "") echo "which hello: nothing (does not see functions)" ;; *) echo "which hello printed: $(which hello)" ;; esac
unset -f hello

cd is a shell builtin
cd is <path>
hello
command -v: not found, status 1
command -v ls gives a path
which hello: nothing (does not see functions)


## 16. The order of expansions
- For each command line, bash does these in order: brace expansion, tilde expansion, parameter / arithmetic / command substitution (left to right), word splitting, pathname (glob) expansion, quote removal.
- Consequence 1: brace expansion happens before variables, so `{1..$n}` does not work. Use `seq` or a `for ((...))` loop.
- Consequence 2: word splitting only touches the results of unquoted expansions, not literal text you typed. Globbing touches unquoted pattern characters wherever they came from, typed or expanded.
- Consequence 3: a quoted `~` is not expanded.

In [37]:
%%bash
n=3
echo {1..3}
echo {1..$n}                 # braces are expanded first, $n is not a number yet
echo "seq works: $(seq 1 $n | paste -sd' ' -)"
echo {a,b,c}.txt
echo ~ | sed "s|$HOME|<HOME>|"
echo "~"  '~'

1 2 3
{1..3}
seq works: 1 2 3
a.txt b.txt c.txt
<HOME>
~ ~


In [38]:
%%bash
touch one.c two.c
star='*'
echo $star.c           # expansion of $star happens first, then globbing: matches the two files
echo "$star.c"         # quoted: stays literal
echo '*'.c             # quoted star: literal
rm -f one.c two.c

one.c two.c
*.c
*.c


## 17. Line continuation and long commands
- A backslash at the very end of a line joins it with the next. There must be nothing (not even a space) after the `\`.
- Inside quotes, a newline is kept as part of the string (a backslash-newline inside double quotes is removed).
- A line ending in `|`, `&&` or `||` continues automatically, no backslash needed.
- Lines inside `( )`, `{ }`, `if ... fi` and so on naturally continue until the construct closes.

In [39]:
%%bash
echo one \
     two \
     three
printf '%s\n' a b c |
  sort -r |
  head -2
true &&
  echo "continued after &&"
echo "double quoted string \
continues here"

one two three
c
b
continued after &&
double quoted string continues here


## 18. Command lists: `;`, `&&`, `||`, `&`, groups
- `a ; b` runs both. `a && b` runs `b` only if `a` succeeded. `a || b` runs `b` only if `a` failed.
- `&&` and `||` have equal precedence and group left to right. So `a && b || c` means `(a && b) || c`.
- Because of that, `a && b || c` is not if/else: if `a` succeeds but `b` fails, `c` also runs.
- `{ a; b; }` groups commands in the current shell (note the space after `{` and the `;` before `}`). `( a; b )` runs them in a subshell, so variables and `cd` inside do not leak out.
- `a &` runs `a` in the background.

In [40]:
%%bash
true  && echo "and: ran because true succeeded"
false || echo "or: ran because false failed"
false && echo "never"   ; echo "; always runs"

and: ran because true succeeded
or: ran because false failed
; always runs


In [41]:
%%bash
# the a && b || c trap
true && false || echo "c ran even though a succeeded (b failed)"
# safe version
if true; then false; else echo "never"; fi; echo "if/else did not run the else branch"

c ran even though a succeeded (b failed)
if/else did not run the else branch


In [42]:
%%bash
x=outer
( x=inner; cd /; echo "in subshell: x=$x" )
echo "after subshell: x=$x"
{ x=group; echo "in group: x=$x"; }
echo "after group: x=$x"

in subshell: x=inner
after subshell: x=outer
in group: x=group
after group: x=group


## 19. Tracing with `set -x`
- `set -x` prints each command (after expansion) to stderr, prefixed by `PS4` (default `+ `). `set +x` turns it off.
- `bash -x script` does the same for a whole script without editing it.
- Combine with a custom `PS4` such as `'+ ${LINENO}: '` to see line numbers.
- Use a fresh shell for demos so tracing does not leak into the rest of the notebook.

In [43]:
%%bash fresh
name="a b"
set -x
echo $name "$name"
n=$((1 + 2))
ls /definitely/missing 2>/dev/null || true
set +x
echo "tracing is off"

+ echo a b 'a b'
a b a b
+ n=3
+ ls /definitely/missing
+ true
+ set +x
tracing is off


In [44]:
%%bash
printf '%s\n' 'x=2' 'echo "x is $x"' > t.sh
bash -x t.sh
rm -f t.sh demo.sh

+ x=2
+ echo 'x is 2'
x is 2


## 20. Putting it together: a safe-by-default skeleton
- Quote expansions, test commands with `command -v`, check exit statuses, and keep constants `readonly`.
- See `04-Scripting-Craft/01-Advanced-Bash` for `set -euo pipefail` and traps, and `03-Functions-And-Arguments` for argument handling.

In [45]:
%%bash fresh
readonly SEP=","
need() { command -v "$1" >/dev/null 2>&1 || { echo "missing tool: $1" >&2; return 1; }; }
need sort && echo "sort is available"
need nosuchtool_xyz || echo "(handled: continuing without it)"
items="b${SEP}a${SEP}c"
printf '%s\n' "$items" | tr "$SEP" '\n' | sort | paste -sd"$SEP" -

sort is available
missing tool: nosuchtool_xyz
(handled: continuing without it)
a,b,c
